# R4-U1a: WavLM head under Protocol A

Select a Colab runtime with a GPU (a T4 is enough; training itself runs on CPU and takes minutes). Sync the new code, `configs/r4_u1_wavlm_head_a.json`, the B cache `data/features/r2_u1_B`, the A gap cache `data/features/r4_u0_A`, the R4-S1 outputs, the frozen manifests and the cohort shards to Drive first. Run the cells in order.

Reads A training and validation embeddings only: no test partition. Trains the **B-selected mean+std head** for 20 epochs, keeps the best selection epoch, freezes `selection.json`, then exports raw predictions for the three validation roles. Nothing is overwritten.

`train` refuses a recipe whose `status` does not record the team's agreement: set it in `configs/r4_u1_wavlm_head_a.json` only after the project owner has reported that agreement.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, subprocess, sys, time
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')  # Change if your project is elsewhere.
OWNER = 'Usman'  # Person running this notebook; written to the ledger.
OUTPUT = ROOT / 'reports/r4/r4_u1_wavlm_A'
BUNDLE = ROOT / 'bundles/r4_u1_wavlm_head_A'
MODEL_CACHE = '/content/hf_cache'  # Pinned WavLM is downloaded here and hash-verified; Colab-local, not synced.
assert (ROOT / 'scripts/r4_u1_wavlm_head.py').is_file(), 'Sync the updated project code to Drive'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src')
os.environ['PYTHONUNBUFFERED'] = '1'
sys.path.insert(0, str(ROOT / 'src'))
print('Output:', OUTPUT, '\nBundle:', BUNDLE)

In [ ]:
import torch, transformers, numpy
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__,
      '| transformers', transformers.__version__, '| numpy', numpy.__version__)
assert torch.cuda.is_available(), 'Choose a GPU runtime (needed for the parity step)'
assert transformers.__version__ == '5.22.0', f'the caches were built with transformers 5.22.0; found {transformers.__version__} (pip install transformers==5.22.0, restart)'
required = ['configs/r4_u1_wavlm_head_a.json', 'configs/r3_u1_wavlm_head_b.json', 'data/features/r2_u1_B/r2_u1_manifest.json',
            'data/features/r4_u0_A/r4_u0_manifest.json', 'data/features/r4_s1_A/crop_bank_A_v1.parquet', 'data/features/r4_s1_A/r4_s1_manifest.json',
            'data/manifests/cohort_v1.parquet', 'data/splits/protocol_A_v1.parquet', 'data/processed/chunks_v1.parquet', 'reports/r1/split_report.json',
            'locks/B_lock_gr3.json']
missing = [p for p in required if not (ROOT / p).is_file()]
assert not missing, f'Missing inputs: {missing}'
assert (ROOT / 'data/processed/cohort_v1').is_dir(), 'Cohort shards are needed for the parity step'
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), 'OUTPUT is not empty; choose a new folder (nothing is overwritten)'
assert not BUNDLE.exists(), 'BUNDLE already exists; choose a new folder'
# The Drive copy must carry the repository's contracts.py (an older copy was used by earlier runs; docstring-only difference).
lock = json.loads((ROOT / 'locks/B_lock_gr3.json').read_text(encoding='utf-8'))
contracts_sha = hashlib.sha256((ROOT / 'src/echoproof/contracts.py').read_bytes()).hexdigest()
assert contracts_sha in json.dumps(lock), f'src/echoproof/contracts.py differs from the GR3 lock ({contracts_sha[:12]}); refresh it from git'

## Fixture tests
Any failure or skip must be resolved before training.

In [ ]:
tests = ['tests/test_r4_u1_wavlm.py', 'tests/test_r4_u0_wavlm_cache.py', 'tests/test_r3_u1_training.py', 'tests/test_r3_u1_adapter.py']
check = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', *tests], text=True, capture_output=True)
print(check.stdout[-3000:], check.stderr[-2000:])
assert check.returncode == 0 and ' skipped' not in check.stdout, 'Resolve test failures or skips before training'

## Alignment, training, selection and export
`train` first proves the A rows equal the A crop bank, the A validation chunks and the split (writes `reports/r4/r4_u1_wavlm_alignment_*.json`; counts must be 35,782 crops and 24,412 chunks), trains the mean+std head for 20 epochs, freezes `selection.json` and exports predictions for the three validation roles. Reading the 60,194-row union takes a few minutes.

In [ ]:
subprocess.run([sys.executable, 'scripts/r4_u1_wavlm_head.py', 'train', '--root', str(ROOT), '--owner', OWNER, '--device', 'cpu'], check=True)

## Direct-waveform parity
Scores 50 selection-role chunks (all five components) from the raw shard audio with the pinned WavLM (TF32 off) and the selected head, and compares them with the cached embeddings (1e-4). A failure stops the notebook.

In [ ]:
subprocess.run([sys.executable, 'scripts/r4_u1_wavlm_head.py', 'parity', '--root', str(ROOT), '--owner', OWNER, '--device', 'cuda',
                '--model-cache', MODEL_CACHE], check=True)

## Package the verified inference bundle
Converts the pinned encoder weights once to safetensors (checked tensor by tensor) and writes the bundle. Uncalibrated: R4-N1 is still required.

In [ ]:
subprocess.run([sys.executable, 'scripts/r4_u1_wavlm_head.py', 'package', '--root', str(ROOT), '--owner', OWNER,
                '--model-cache', MODEL_CACHE], check=True)

## Verify the handoff and write the report
Checks recorded hashes, the completed ledger row, the latest parity result and that the packaged bundle loads and scores a waveform. It does not replace independent review.

In [ ]:
import numpy as np
from echoproof.models.wavlm_adapter import WavLMBundleAdapter
def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1048576), b''):
            h.update(block)
    return h.hexdigest()
evidence = json.loads((OUTPUT / 'evidence.json').read_text(encoding='utf-8'))
assert evidence['status'] == 'validation_predictions_complete'
assert sha256(OUTPUT / 'selection.json') == evidence['selection_sha256']
assert all(sha256(OUTPUT / name) == digest for name, digest in evidence['output_sha256'].items())
assert sha256(OUTPUT / 'run_mean_std' / 'best.pt') == evidence['selection']['checkpoint_sha256']['mean_std']
with (ROOT / 'reports/ledger.csv').open(newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['run_id'].startswith('r4_u1_wavlm_head_A') and r['status'] == 'completed']
assert len(rows) >= 1, 'expected a completed R4-U1a ledger row'
parity_path = sorted((ROOT / 'reports/r4').glob('r4_u1_wavlm_parity_*.json'))[-1]
parity = json.loads(parity_path.read_text(encoding='utf-8'))
assert parity['passed'] is True and parity['pooling'] == 'mean_std'
adapter = WavLMBundleAdapter.load(BUNDLE, device='cuda')
probe = adapter.score_chunks([np.random.default_rng(0).normal(0, 0.05, 16000).astype(np.float32)])
assert probe.shape == (1,) and 0 <= probe[0] <= 1
print('selected epoch:', evidence['selection']['selected_epoch'], '| selection macro F1:', evidence['selection']['selection_macro_f1'],
      '| parity max |dp|:', parity['max_abs_probability_difference'])
report = {'task': 'R4-U1a', 'output': str(OUTPUT), 'bundle': str(BUNDLE), 'evidence_sha256': sha256(OUTPUT / 'evidence.json'),
          'parity_report': parity_path.name, 'parity_report_sha256': sha256(parity_path), 'completed_ledger_rows': len(rows),
          'selection_sha256': evidence['selection_sha256'], 'counts': evidence['counts'], 'selection_panels': evidence['selection_panels'],
          'torch': torch.__version__, 'transformers': transformers.__version__, 'gpu': torch.cuda.get_device_name(0)}
path = ROOT / 'reports/r4' / ('r4_u1_wavlm_colab_' + time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()) + '.json')
assert not path.exists()
path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
print('report:', path)